# Learning curves, resolving power and noise floor

Work plan step 5, item 5. This notebook **contains no logic** — it reads the
JSONL logs of the driver scripts and calls `twinlink_ml.evaluation.plots`. Whoever wants to
change a figure changes the module, not this cell; otherwise two
definitions of the same curve exist.

The logs are generated with:

```bash
python scripts/study_variance_sources.py  --log artifacts/runs/variance.jsonl
python scripts/study_noise_level.py       --log artifacts/runs/noise.jsonl
python scripts/study_validation_policy.py --log artifacts/runs/policy.jsonl
```

Every learning curve carries four elements, and none of them is decoration:

1. **Median over the replicates** with a min/max band (not ±1σ — with three replicates
   the normality assumption cannot be checked).
2. **The persistence baseline as a horizontal line.** It is not trained; over
   $n_\mathrm{train}$ it is constant.
3. **Resolution threshold** as a red band. Nothing inside it is a result.
4. **Noise floor** as the lower asymptote — the level at which a curve *must* flatten.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from twinlink_ml.evaluation.plots import (
    plot_learning_curve,
    plot_noise_study,
    plot_variance_decomposition,
)
from twinlink_ml.evaluation.resolution import (
    resolution_report,
    resolution_thresholds,
    select,
)
from twinlink_ml.utils import read_jsonl

RUNS = Path("..") / "artifacts" / "runs"
HORIZON = 24
TARGETS = ("energy_target", "product_moisture_kg_per_kg", "d50_um")

variance = read_jsonl(RUNS / "variance.jsonl")
print(len(variance), "Einträge")

## 1 Spread by source

`data` = dataset realisation (excitation and initial states), `noise` = sensor noise,
`model` = initialisation and batch order, `joint` = all three together via one
master seed.

For the baselines `model` is exactly zero — they have no stochastic fitting. This
is the regression check: if the bar were not zero, there would be randomness somewhere that
does not belong there.

In [ ]:
fig, axes = plt.subplots(len(TARGETS), 1, figsize=(9, 11), constrained_layout=True)
for axis, target in zip(axes, TARGETS, strict=True):
    plot_variance_decomposition(variance, target=target, horizon=HORIZON, ax=axis)
fig

In [ ]:
for target in TARGETS:
    print(resolution_report(variance, target=target, horizon=HORIZON))
    print()

## 2 Learning curves with resolution threshold

The threshold comes from the seed spread of the **parameter-free** persistence baseline and is
therefore independent of $n_\mathrm{train}$ — it is a lower bound of the comparison.
The additional estimation spread at small $n$ is in the table above under `static`
and `arx`.

In [ ]:
thresholds = {
    target: resolution_thresholds(variance, target=target, horizon=HORIZON) for target in TARGETS
}

fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 4.4), constrained_layout=True)
for axis, target in zip(axes, TARGETS, strict=True):
    plot_learning_curve(
        variance,
        target=target,
        horizon=HORIZON,
        resolution=thresholds[target],
        ax=axis,
        title=target,
    )
fig

## 3 Noise level and saturation level

The green line is the **model-free** noise floor: the error that a perfect
predictor achieves on the same windows. The gap between curve and floor is
everything a better model can still gain at all.

In [ ]:
noise = read_jsonl(RUNS / "noise.jsonl")
levels = sorted({float(r.params["noise/relative_std"]) for r in noise})
print("Rauschniveaus:", levels)

In [ ]:
fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 4.4), constrained_layout=True)
for axis, target in zip(axes, TARGETS, strict=True):
    plot_noise_study(noise, {}, target=target, horizon=HORIZON, n_train=64, ax=axis)
fig

The noise floors themselves are not in the records log — they are not a
model property and would be misfiled under a model name.
`run_noise_study` returns them as its second return value; the numbers are in
`04_stand_ap1.md`, Befund 13.

## 4 Learning curve per noise level

**Filter** before decomposing: if two noise levels are in the same call, the
"spread over seeds" would in truth be the effect of the noise level. `collect_spreads` detects
this and raises — `select` is the right way.

In [ ]:
fig, axes = plt.subplots(1, len(levels), figsize=(16, 4.4), constrained_layout=True)
for axis, level in zip(axes, levels, strict=True):
    subset = select(noise, **{"noise/relative_std": level})
    plot_learning_curve(
        subset,
        target="product_moisture_kg_per_kg",
        horizon=HORIZON,
        ax=axis,
        title=f"Produktfeuchte, sigma_rel = {level}",
    )
fig

## 5 Validation mode

`FIXED_POOL` versus `NONE`. For all three baselines the difference is exactly zero — none
uses the validation split. A non-zero difference would be the proof that
validation runs leak into training; the substantive number arises with the first
early-stopping model in step 6.

In [ ]:
from twinlink_ml.evaluation import summarise_policy_gap

policy = read_jsonl(RUNS / "policy.jsonl")
gaps = summarise_policy_gap(policy, target="energy_target", horizon=HORIZON)
for (label, n_train), (fixed, none, delta) in gaps.items():
    print(f"{label:>12} n={n_train:>3}  FIXED_POOL {fixed:.4f}  NONE {none:.4f}  D {delta:+.2e}")

## 6 Architecture comparison and observability

From `scripts/study_architectures.py`. **Two charts, not one**: the primary metric
differs between the operating modes, because without inline measurement the level of the
run is part of the task (see `evaluation.protocol.primary_metric`). A joint
chart would have two different y-axes under one name.

In [ ]:
from twinlink_ml.evaluation import ObservationMode, primary_metric
from twinlink_ml.evaluation.resolution import architecture_verdicts

architectures = read_jsonl(RUNS / "architectures.jsonl")

for mode in (ObservationMode.INLINE, ObservationMode.SOFT_SENSOR):
    subset = select(architectures, **{"observation/mode": mode.value})
    metric = primary_metric(mode)
    reference = "persistence" if mode is ObservationMode.INLINE else "climatology"
    fig, axes = plt.subplots(1, len(TARGETS), figsize=(16, 4.4), constrained_layout=True)
    for axis, target in zip(axes, TARGETS, strict=True):
        plot_learning_curve(
            subset,
            target=target,
            horizon=HORIZON,
            metric=metric,
            resolution=resolution_thresholds(
                subset,
                target=target,
                horizon=HORIZON,
                reference_label=reference,
                metric=metric,
            ),
            ax=axis,
            title=f"{target} ({mode.value})",
        )
    display(fig)
    plt.close(fig)

The verdicts deliberately do not come as a bare percentage: `architecture_verdicts`
returns the difference **only together with** the resolution threshold and phrases the
result the way it may be reported.

In [ ]:
for mode in (ObservationMode.INLINE, ObservationMode.SOFT_SENSOR):
    subset = select(architectures, **{"observation/mode": mode.value})
    metric = primary_metric(mode)
    free = "persistence" if mode is ObservationMode.INLINE else "climatology"
    reference = "arx" if mode is ObservationMode.SOFT_SENSOR else free
    print(f"--- {mode.value}: product_moisture gegen {reference} ---")
    for verdict in architecture_verdicts(
        subset,
        target="product_moisture_kg_per_kg",
        horizon=HORIZON,
        reference_label=reference,
        resolution_label=free,
        metric=metric,
    ):
        print(f"  n={verdict.n_train:>3}  {verdict.statement}")